## Aviso metodológico

> Análisis histórico deprecado del flujo principal de la tesina. Se conserva para trazabilidad. Sus métodos y resultados no forman parte de la especificación activa ni de las conclusiones finales.


# 11. Diseño muestral formal e inferencia descriptiva

Esta libreta implementa una revisión acotada del diseño muestral ENIGH antes de modelar. Usa los marts nominales de `revision_4`, conserva los años separados y calcula estadística descriptiva ponderada con errores estándar aproximados por JKn estratificado por UPM.

La etapa no reabre la homologación monetaria ni la discrepancia documentada con Banxico. Los resultados monetarios se reportan en pesos nominales trimestrales.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.diseno_muestral_jkn import build_stage11_outputs, dependency_versions

manifest = build_stage11_outputs(PROJECT_ROOT)
print("Proyecto:", PROJECT_ROOT)
print("Estado global:", manifest["verificacion"]["estado_global"])
print("Validación R/survey:", manifest["verificacion"]["validacion_R_survey"])
print("Motor de figuras:", [f["motor"] for f in manifest["salidas"]["figures"]])
print("Dependencias:", dependency_versions())

Proyecto: C:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling
Estado global: ok
Validación R/survey: pendiente_entorno
Motor de figuras: ['svg_fallback', 'svg_fallback']
Dependencias: {'python': '3.12.14', 'python_executable': 'C:\\Users\\lucia\\.cache\\codex-runtimes\\codex-primary-runtime\\dependencies\\python\\python.exe', 'platform': 'Windows-11-10.0.26200-SP0', 'pandas': '3.0.1', 'numpy': '2.3.5', 'scipy': 'no instalado', 'matplotlib': 'no instalado', 'seaborn': 'no instalado', 'nbformat': 'no instalado', 'nbclient': 'no instalado', 'pypdf': 'instalado', 'Rscript': 'no disponible'}


## Estimandos

Para cada dominio descriptivo `D` se usan:

`N_hat_D = sum_i w_i d_i`

`Y_hat_D = sum_i w_i d_i y_i`

`mu_hat_D = Y_hat_D / N_hat_D`

`p_hat_D = sum_i w_i d_i I(y_i > 0) / sum_i w_i d_i`

El contraste Norte-Sur se calcula como `Delta_hat = mu_hat_Norte - mu_hat_Sur`, recalculando ambas medias dentro de cada réplica.

## JKn y referencia t

Para cada estrato `h` con `m_h` UPMs, la réplica elimina una UPM y multiplica el resto del estrato por `m_h / (m_h - 1)`. La varianza se centra en la estimación completa:

`V_JK = sum_h ((m_h - 1) / m_h) sum_j (theta_(h,j) - theta_hat)^2`

Los intervalos usan `t(0.975, nu)` con `nu_D = M_D - H_D` por soporte del dominio. Esta es una referencia aproximada para la distribución muestral studentizada, no una prueba de t exacta clásica.

In [2]:
table_dir = PROJECT_ROOT / "reports" / "tables" / "diseno_muestral"
audit = pd.read_csv(table_dir / "audit_diseno_anio_unidad.csv")
cols = ["anio", "unidad", "filas", "estratos", "upm", "estratos_singleton", "upm_raw_en_multiples_estratos", "nu_diseno_M_menos_H", "estado"]
print(audit[cols].to_string(index=False))

 anio  unidad  filas  estratos   upm  estratos_singleton  upm_raw_en_multiples_estratos  nu_diseno_M_menos_H estado
 2018   hogar  74647       543  8377                   0                              0                 7834     ok
 2018 persona 269206       543  8377                   0                              0                 7834     ok
 2020   hogar  89006       558 10118                   0                              0                 9560     ok
 2020 persona 315743       558 10118                   0                              0                 9560     ok
 2022   hogar  90102       560 10211                   0                              0                 9651     ok
 2022 persona 309684       560 10211                   0                              0                 9651     ok
 2024   hogar  91414       681 10569                   0                              0                 9888     ok
 2024 persona 308598       681 10569                   0                

In [3]:
consistency = pd.read_csv(table_dir / "consistencia_hogar_persona.csv")
print(consistency.to_string(index=False))

 anio  hogares_mart_hogar  hogares_en_mart_persona  hogares_comunes  hogares_persona_sin_hogar  hogares_hogar_sin_persona  hogares_con_diseno_variable_en_persona  hogares_con_diferencia_factor  hogares_con_diferencia_factor_hogar  hogares_con_diferencia_est_dis  hogares_con_diferencia_upm estado
 2018               74647                    74647            74647                          0                          0                                       0                              0                                    0                               0                           0     ok
 2020               89006                    89006            89006                          0                          0                                       0                              0                                    0                               0                           0     ok
 2022               90102                    90102            90102                          0               

In [4]:
estimates = pd.read_csv(table_dir / "estimaciones_nacionales_regionales.csv")
national = estimates[estimates["dominio"].eq("Nacional")].copy()
cols = ["anio", "unidad", "estimando", "n_muestral", "poblacion_expandida", "M_D", "H_D", "estimacion", "se_jkn", "nu", "ic95_inf", "ic95_sup", "estado"]
print(national[cols].to_string(index=False))

 anio  unidad                                   estimando  n_muestral  poblacion_expandida   M_D  H_D   estimacion     se_jkn   nu     ic95_inf     ic95_sup estado
 2018   hogar               media_ingreso_corriente_hogar       74647           34400515.0  8377  543 49851.005352 454.970812 7834 48959.141152 50742.869552     ok
 2020   hogar               media_ingreso_corriente_hogar       89006           35749659.0 10118  558 50309.313150 400.486218 9560 49524.275195 51094.351106     ok
 2022   hogar               media_ingreso_corriente_hogar       90102           37560123.0 10211  560 63695.457900 440.348067 9651 62832.283295 64558.632505     ok
 2024   hogar               media_ingreso_corriente_hogar       91414           38830230.0 10569  681 77863.843012 605.613814 9888 76676.716435 79050.969589     ok
 2018 persona proporcion_ingreso_laboral_negocio_positivo      269206          123934029.0  8377  543     0.478346   0.001505 7834     0.475396     0.481297     ok
 2018 persona   

In [5]:
regional = estimates[
    estimates["estimando"].isin([
        "media_ingreso_corriente_hogar",
        "media_ingreso_laboral_negocio_positivo",
    ])
    & ~estimates["dominio"].eq("Nacional")
].copy()
cols = ["anio", "unidad", "dominio", "estimando", "estimacion", "se_jkn", "nu", "ic95_inf", "ic95_sup", "estado"]
print(regional[cols].to_string(index=False))

 anio  unidad      dominio                              estimando   estimacion      se_jkn   nu     ic95_inf      ic95_sup estado
 2018   hogar        Norte          media_ingreso_corriente_hogar 58698.970578 1094.690614 2178 56552.223415  60845.717740     ok
 2018   hogar Centro Norte          media_ingreso_corriente_hogar 52175.754028  700.233909 2282 50802.592472  53548.915584     ok
 2018   hogar       Centro          media_ingreso_corriente_hogar 52910.582150  961.469132 1924 51024.951063  54796.213236     ok
 2018   hogar          Sur          media_ingreso_corriente_hogar 35063.652503  475.046474 1450 34131.800685  35995.504321     ok
 2020   hogar        Norte          media_ingreso_corriente_hogar 62594.345292 1174.162955 2647 60291.975416  64896.715167     ok
 2020   hogar Centro Norte          media_ingreso_corriente_hogar 52504.774028  602.136745 2782 51324.094020  53685.454036     ok
 2020   hogar       Centro          media_ingreso_corriente_hogar 51086.409038  783.822814

In [6]:
contrast = pd.read_csv(table_dir / "contraste_norte_sur.csv")
cols = ["anio", "estimacion", "se_jkn", "nu", "ic95_inf", "ic95_sup", "media_a", "media_b", "estado"]
print(contrast[cols].to_string(index=False))

 anio   estimacion      se_jkn   nu     ic95_inf     ic95_sup      media_a      media_b estado
 2018 12339.248042  594.830997 3619 11173.010668 13505.485416 25861.427146 13522.179104     ok
 2020 13648.569250  607.606183 4408 12457.355929 14839.782571 27370.759794 13722.190544     ok
 2022 15986.300109  542.247154 4455 14923.226394 17049.373824 35026.445765 19040.145656     ok
 2024 20729.588267 1387.762785 4506 18008.892382 23450.284151 43273.998102 22544.409835     ok


In [7]:
validations = pd.read_csv(table_dir / "validaciones_diseno_muestral.csv")
print(validations.to_string(index=False))

                                         validacion         resultado  diferencia_maxima   tolerancia                                                                        detalle
            estimacion_vs_calculo_directo_ponderado                ok       7.275958e-12 1.000000e-07 Las estimaciones puntuales del JKn coinciden con el calculo ponderado directo.
      jkn_agregado_vs_replicas_explicitas_media_toy                ok       7.105427e-15 1.000000e-10               La agregacion por estrato-UPM reproduce las replicas explicitas.
jkn_dominio_con_upm_cero_vs_replicas_explicitas_toy                ok       1.421085e-14 1.000000e-10              El dominio preserva UPMs fuera del dominio con contribucion cero.
  jkn_agregado_vs_replicas_explicitas_contraste_toy                ok       1.065814e-14 1.000000e-10                    El contraste recalcula ambas medias dentro de cada replica.
             invariancia_medias_se_al_escalar_pesos                ok       4.440892e-15 1.0000

In [8]:
print("Figuras generadas:")
for fig in manifest["salidas"]["figures"]:
    print(f"- {fig['archivo']} ({fig['motor']})")

Figuras generadas:
- C:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\reports\figures_documentacion\diseno_muestral_ingreso_hogar_ic95.svg (svg_fallback)
- C:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\reports\figures_documentacion\diseno_muestral_contraste_norte_sur_ic95.svg (svg_fallback)


## Lectura metodológica

Todos los años tienen `factor`, `est_dis` y `upm` completos, sin estratos singleton en el diseño observable. Las llaves hogar/persona son consistentes entre marts.

Las estimaciones son descriptivas y asociativas. Las diferencias nominales entre años no deben leerse como cambios reales de poder adquisitivo. La comparación especializada con R `survey` queda preparada en `reports/tables/diseno_muestral/validacion_r_survey_jkn.R`, pero no se ejecuta en este entorno porque `Rscript` no está disponible.